In [1]:
import os
import pandas as pd
from fuzzywuzzy import process

# === CONFIG ===
ward_reference_file = "official_wards.csv"  # from OS Boundary-Line
input_files = "input_files"
output_files = "processed_files"
match_threshold = 85  # fuzzy match confidence


# === HELPERS ===
def normalize_name(name):
    if not isinstance(name, str):
        return ""
    name = (
        name.lower()
        .replace("&", "and")
        .replace("-", " ")
        .replace("ward", "")
        .replace("council", "")
        .strip()
    )
    return " ".join(name.split())


def best_match(name, ref_list, threshold):
    """Return best fuzzy match if above threshold."""
    if not name:
        return None, 0
    # ensure ref_list is a Python list
    ref_list = list(ref_list)
    result = process.extractOne(name, ref_list)
    if result is None:
        return None, 0
    # unpack dynamically: first two items
    match = result[0]
    score = result[1]
    if score >= threshold:
        return match, score
    else:
        return None, score

    
input_dir = "scotland_data_code_attempt"
output_dir = "processed_files"
match_threshold = 85  # fuzzy match confidence

In [2]:
# Import ward file (filter for Scotland!)
wards_ref = pd.read_csv('wardcodelist.csv')
wards_ref["clean_name"] = wards_ref["ward_name"].apply(normalize_name)
wards_ref = wards_ref.drop(columns=['WD25NMW','ObjectId'])
wards_ref = wards_ref[wards_ref['ward_code'].str.startswith("S", na=False)]

In [3]:
"""
# === RECURSIVE FILE PROCESSING ===
for root, dirs, files in os.walk(input_dir):
    # Compute relative path to preserve subdirs
    rel_path = os.path.relpath(root, input_dir)
    output_subdir = os.path.join(output_dir, rel_path)
    os.makedirs(output_subdir, exist_ok=True)

    for file_name in files:
        if not file_name.lower().endswith((".csv", ".txt", ".blt")):
            continue

        input_path = os.path.join(root, file_name)

        # Read all lines at once
        with open(input_path, "r", encoding="utf-8", errors="ignore") as f:
            lines = [line.rstrip("\n") for line in f if line.strip()]

        if not lines:
            print(f"⚠️ {os.path.join(rel_path, file_name)}: empty file")
            continue

        # Last line is the ward title
        ward_title = lines[-1]
        clean_title = normalize_name(ward_title)
        match, score = best_match(clean_title, wards_ref["clean_name"], match_threshold)

        if match:
            ward_code = wards_ref.loc[wards_ref["clean_name"] == match, "ward_code"].iloc[0]
            # Replace last line with ward code
            lines[-1] = ward_code

            # Save file to mirrored subdir with ward_code as filename
            new_file_name = f"{ward_code}{os.path.splitext(file_name)[1]}"
            output_path = os.path.join(output_subdir, new_file_name)
            with open(output_path, "w", encoding="utf-8") as f:
                f.write("\n".join(lines) + "\n")

        else:
            print(f"⚠️ {os.path.join(rel_path, file_name)}: no ward match found")

print("\nAll done. Files saved to:", output_dir)
"""

'\n# === RECURSIVE FILE PROCESSING ===\nfor root, dirs, files in os.walk(input_dir):\n    # Compute relative path to preserve subdirs\n    rel_path = os.path.relpath(root, input_dir)\n    output_subdir = os.path.join(output_dir, rel_path)\n    os.makedirs(output_subdir, exist_ok=True)\n\n    for file_name in files:\n        if not file_name.lower().endswith((".csv", ".txt", ".blt")):\n            continue\n\n        input_path = os.path.join(root, file_name)\n\n        # Read all lines at once\n        with open(input_path, "r", encoding="utf-8", errors="ignore") as f:\n            lines = [line.rstrip("\n") for line in f if line.strip()]\n\n        if not lines:\n            print(f"⚠️ {os.path.join(rel_path, file_name)}: empty file")\n            continue\n\n        # Last line is the ward title\n        ward_title = lines[-1]\n        clean_title = normalize_name(ward_title)\n        match, score = best_match(clean_title, wards_ref["clean_name"], match_threshold)\n\n        if match

In [5]:
file = 'raw_census_data/uv501_education.csv'
converting_db = pd.read_csv(file)
converting_db.head()

for idx, row in converting_db.iterrows():
    clean_title = normalize_name(row["Electoral Ward 2022"])
    match, score = best_match(clean_title, wards_ref["clean_name"], match_threshold)
    if match:
        ward_code = wards_ref.loc[wards_ref["clean_name"] == match, "ward_code"].iloc[0]
        converting_db.at[idx, "Electoral Ward 2022"] = ward_code
converting_db["pct_uni_educated"] = round(converting_db["Degree level qualifications or above"]/converting_db["All people aged 16 and over"],4)
converting_db=converting_db[["Electoral Ward 2022","pct_uni_educated"]]
converting_db.to_csv(file, index=False)